<a href="https://colab.research.google.com/github/luciacardozo472/TRABAJOP_IA/blob/main/multiagentes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Instalacion de dependencias

In [ ]:
!pip install -q transformers sentence-transformers scikit-learn pandas numpy torch

Dataset CSV - Empleados (sin normalizar)

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 500

departamentos = ['IT', 'Ventas', 'RRHH', 'Gerencia', 'Marketing', 'Finanzas']
satisfacciones = ['alta', 'media', 'baja']

edades = np.random.randint(22, 60, n)
experiencias = np.random.randint(0, 35, n)
departamento = np.random.choice(departamentos, n)

# Salario correlacionado con experiencia y departamento
salario_base = {'IT': 55000, 'Ventas': 38000, 'RRHH': 42000,
                'Gerencia': 90000, 'Marketing': 45000, 'Finanzas': 60000}
salarios = np.array([salario_base[d] + experiencias[i] * 1500 + np.random.randint(-5000, 5000)
                     for i, d in enumerate(departamento)])

# Satisfaccion correlacionada con salario
satisfaccion = []
for s in salarios:
    if s > 75000:
        satisfaccion.append(np.random.choice(['alta', 'media'], p=[0.75, 0.25]))
    elif s > 50000:
        satisfaccion.append(np.random.choice(['alta', 'media', 'baja'], p=[0.4, 0.4, 0.2]))
    else:
        satisfaccion.append(np.random.choice(['media', 'baja'], p=[0.35, 0.65]))

df_raw = pd.DataFrame({
    'edad': edades,
    'salario': salarios,
    'departamento': departamento,
    'experiencia': experiencias,
    'satisfaccion': satisfaccion
})

# Introducir valores nulos (~5%)
for col in ['salario', 'experiencia', 'satisfaccion']:
    idx = np.random.choice(df_raw.index, size=int(n * 0.05), replace=False)
    df_raw.loc[idx, col] = np.nan

print(f'Dataset generado: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas')
print(f'Valores nulos: {df_raw.isnull().sum().sum()}')
print(f'Distribucion satisfaccion:')
print(df_raw['satisfaccion'].value_counts())
df_raw.head(10)

Dataset generado: 500 filas x 5 columnas
Valores nulos: 75
Distribucion satisfaccion:
satisfaccion
alta     263
media    153
baja      59
Name: count, dtype: int64


,edad,salario,departamento,experiencia,satisfaccion
0,50,57488.0,Finanzas,1.0,alta
1,36,43116.0,Ventas,4.0,baja
2,29,81446.0,Ventas,28.0,media
3,42,81179.0,IT,18.0,alta
4,40,52182.0,Marketing,7.0,alta
5,44,42626.0,RRHH,0.0,media
6,32,78924.0,Marketing,21.0,alta
7,32,61933.0,RRHH,16.0,alta
8,45,NaN,Gerencia,6.0,alta
9,57,77980.0,Marketing,24.0,media


---
## AGENTE 1 - Normalizador

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

class AgenteNormalizador:
    def __init__(self):
        self.scaler = StandardScaler()
        self.label_encoders = {}
        self.log = []

    def limpiar(self, df):
        nulos = df.isnull().sum().sum()
        df = df.drop_duplicates()
        self.log.append(f'[Limpieza] Nulos encontrados: {nulos}')
        return df

    def imputar(self, df):
        for col in df.columns:
            if df[col].isnull().any():
                if df[col].dtype in ['float64', 'int64']:
                    val = df[col].median()
                    df[col] = df[col].fillna(val)
                    self.log.append(f'[Imputacion] {col} -> mediana ({val:.1f})')
                else:
                    val = df[col].mode()[0]
                    df[col] = df[col].fillna(val)
                    self.log.append(f'[Imputacion] {col} -> moda ({val})')
        return df

    def codificar(self, df, cols):
        for col in cols:
            le = LabelEncoder()
            df[col + '_enc'] = le.fit_transform(df[col].astype(str))
            self.label_encoders[col] = le
            self.log.append(f'[Codificacion] {col} -> clases: {list(le.classes_)}')
        return df

    def escalar(self, df, cols):
        df[cols] = self.scaler.fit_transform(df[cols])
        self.log.append(f'[Escalado] StandardScaler en: {cols}')
        return df

    def procesar(self, df):
        print('=' * 55)
        print('AGENTE 1 - Normalizador')
        print('=' * 55)
        df = self.limpiar(df.copy())
        df = self.imputar(df)
        df = self.codificar(df, ['departamento', 'satisfaccion'])
        df = self.escalar(df, ['edad', 'salario', 'experiencia'])
        for e in self.log:
            print(' ', e)
        print(f'\nDataset limpio: {df.shape}')
        return df

agente1 = AgenteNormalizador()
df_limpio = agente1.procesar(df_raw)
df_limpio[['edad', 'salario', 'experiencia', 'departamento_enc', 'satisfaccion_enc']].head()

AGENTE 1 - Normalizador
  [Limpieza] Nulos encontrados: 75
  [Imputacion] salario -> mediana (78791.0)
  [Imputacion] experiencia -> mediana (18.0)
  [Imputacion] satisfaccion -> moda (alta)
  [Codificacion] departamento -> clases: ['Finanzas', 'Gerencia', 'IT', 'Marketing', 'RRHH', 'Ventas']
  [Codificacion] satisfaccion -> clases: [np.str_('alta'), np.str_('baja'), np.str_('media')]
  [Escalado] StandardScaler en: ['edad', 'salario', 'experiencia']

Dataset limpio: (500, 7)


,edad,salario,experiencia,departamento_enc,satisfaccion_enc
0,0.786010,-0.964922,-1.612906,0,0
1,-0.482040,-1.619880,-1.314441,5,1
2,-1.116065,0.126886,1.073281,5,2
3,0.061410,0.114719,0.078397,2,0
4,-0.119740,-1.206726,-1.015976,3,0
